This notebook handles the one-time offline export of Community-1's pretrained weights into runtime-friendly formats. 

- export the segmentation model (PyanNet) and embedding model (WeSpeakerResNet34) to both ONNX and OpenVINO IR
- Validate each exported model against the original PyTorch outputs. 

Running this notebook is a prerequisite to using the inference pipeline; it requires a valid HuggingFace token but produces artifacts with zero pyannote runtime dependency.

In [1]:
import os
import dotenv

dotenv.load_dotenv("..")
HUGGINGFACE_ACCESS_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HUGGINGFACE_ACCESS_TOKEN)

In [17]:
import torch
import numpy as np

MODELS_FOLDER = "../models"
os.makedirs(MODELS_FOLDER, exist_ok=True)

for model_type in ["segmentation", "embedding"]:
    os.makedirs(f"{MODELS_FOLDER}/onnx/{model_type}", exist_ok=True)
    os.makedirs(f"{MODELS_FOLDER}/openvino/{model_type}", exist_ok=True)

In [7]:
pipeline._inferences['_segmentation'].model.lstm.flatten_parameters()

seg_model = pipeline._inferences['_segmentation'].model
seg_model.eval().cpu()

emb_model = pipeline._inferences['_embedding'].model_
emb_model.eval().cpu()

print("models loaded...")

models loaded...


## 1 - Export the Segmentation Model

In [ ]:
dummy_audio = torch.randn(1, 1, 160000)

segmentation_onnx_output = f"{MODELS_FOLDER}/onnx/segmentation/model.onnx"

torch.onnx.export(
    seg_model,
    dummy_audio,
    segmentation_onnx_output,
    input_names=["audio"],
    output_names=["segmentation"],
    dynamic_axes={
        "audio": {2: "num_samples"},
        "segmentation": {1: "num_frames"}
    },
    opset_version=17,
    dynamo=False,
    do_constant_folding=True,
    export_params=True,
)

In [ ]:
import openvino as ov

segmentation_ov_output = f"{MODELS_FOLDER}/openvino/segmentation/model.xml"

core = ov.Core()
ov_seg = core.compile_model(
    ov.convert_model(segmentation_onnx_output),
    device_name="CPU"
)

ov.save_model(
    ov.convert_model(segmentation_onnx_output),
    segmentation_ov_output
)

In [ ]:
class ResNetWrapper(torch.nn.Module):
    def __init__(self, resnet):
        super().__init__()
        self.resnet = resnet

    def forward(self, fbank: torch.Tensor) -> torch.Tensor:
        _, embeddings = self.resnet(fbank, weights=None)
        return embeddings


with torch.no_grad():
    dummy_wave = torch.randn(1, 1, 16000)
    fbank = emb_model.compute_fbank(dummy_wave)
    print(f"fbank shape: {fbank.shape}")


wrapper = ResNetWrapper(emb_model.resnet)
wrapper.eval()
dummy_fbank = torch.randn(1, fbank.shape[1], fbank.shape[2])

fbank shape: torch.Size([1, 98, 80])


In [ ]:
embedding_onnx_output = f"{MODELS_FOLDER}/onnx/embedding/model.onnx"

torch.onnx.export(
    wrapper,
    dummy_fbank,
    embedding_onnx_output,
    input_names=["fbank"],
    output_names=["embedding"],
    dynamic_axes={
        "fbank": {0: "batch", 1: "num_frames"},
        "embedding": {0: "batch"}
    },
    opset_version=17,
    dynamo=False,
    do_constant_folding=True,
    export_params=True,
)

In [ ]:
embedding_ov_output = f"{MODELS_FOLDER}/openvino/embedding/model.xml"

ov.save_model(
    ov.convert_model(embedding_onnx_output),
    embedding_ov_output
)

## 3 - Test Inference

In [40]:
import onnxruntime as ort
import numpy as np


results = {}

dummy_wav = np.random.randn(1, 1, 160000).astype(np.float32)
dummy_mel = np.random.randn(1, fbank.shape[1], fbank.shape[2]).astype(np.float32)


print("\n── ONNX Runtime ──")
seg_sess = ort.InferenceSession(segmentation_onnx_output)
results["seg-onnx"] = out = seg_sess.run(None, {"audio": dummy_wav})
print(f"  segmentation: {out[0].shape}")

emb_sess = ort.InferenceSession(embedding_onnx_output)
results["emb-onnx"] = out = emb_sess.run(None, {"fbank": dummy_mel})
print(f"  embedding:    {out[0].shape}")


print("\n── OpenVINO ──")
core = ov.Core()

ov_seg = core.compile_model(segmentation_ov_output, "CPU")
results["seg-opv"] = out = ov_seg({"audio": dummy_wav})
print(f"  segmentation: {list(out.values())[0].shape}")

ov_emb = core.compile_model(embedding_ov_output, "CPU")
results["emb-opv"] = out = ov_emb({"fbank": dummy_mel})
print(f"  embedding:    {list(out.values())[0].shape}")



── ONNX Runtime ──
  segmentation: (1, 589, 7)
  embedding:    (1, 256)

── OpenVINO ──
  segmentation: (1, 589, 7)
  embedding:    (1, 256)


In [41]:
results["seg-onnx"][0][0,1]

array([-0.05394956, -6.599312  , -4.9206953 , -3.1815968 , -8.472809  ,
       -7.9076734 , -6.3376226 ], dtype=float32)

In [42]:
results["seg-opv"]["segmentation"][0,1]

array([-0.05386428, -6.6032524 , -4.922359  , -3.1830096 , -8.475631  ,
       -7.9105434 , -6.3394127 ], dtype=float32)

In [53]:
assert np.allclose(results["seg-onnx"][0], results["seg-opv"]["segmentation"], rtol=1e-2)

In [57]:
results["emb-opv"][0].shape

(1, 256)

In [58]:
results["emb-opv"]["embedding"].shape

(1, 256)

In [59]:
assert np.allclose(results["emb-opv"][0], results["emb-opv"]["embedding"], rtol=1e-2)


# 4 - Runtime Benchmark

In [60]:
import time
import numpy as np
import onnxruntime as ort
import openvino as ov

N_RUNS = 1000  # warm up + measure
WARMUP = 5

def benchmark(fn, label, n_runs=N_RUNS, warmup=WARMUP):
    # Warmup
    for _ in range(warmup):
        fn()
    # Measure
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    arr = np.array(times) * 1000  # ms
    print(f"  {label:<35} mean={arr.mean():.2f}ms  std={arr.std():.2f}ms  min={arr.min():.2f}ms  max={arr.max():.2f}ms")
    return arr

In [61]:
# ── Segmentation ──────────────────────────────────────────────────────────────
print("\n── Segmentation (10s chunk · input: 1×1×160000) ──")

seg_onnx = ort.InferenceSession(segmentation_onnx_output)
seg_ov   = ov.Core().compile_model(segmentation_ov_output, "CPU")

results = {}
results["seg_onnx"] = benchmark(
    lambda: seg_onnx.run(None, {"audio": dummy_wav}),
    "ONNX  segmentation"
)
results["seg_opv"] = benchmark(
    lambda: seg_ov({"audio": dummy_wav}),
    "OpenVINO  segmentation"
)


── Segmentation (10s chunk · input: 1×1×160000) ──
  ONNX  segmentation                  mean=40.59ms  std=8.56ms  min=27.97ms  max=119.86ms
  OpenVINO  segmentation              mean=48.40ms  std=5.80ms  min=36.08ms  max=77.24ms


In [62]:
# ── Embedding ─────────────────────────────────────────────────────────────────
print("\n── Embedding (1s chunk · input: 1×98×80) ──")

emb_onnx = ort.InferenceSession(embedding_onnx_output)
emb_ov   = ov.Core().compile_model(embedding_ov_output, "CPU")

results["emb_onnx"] = benchmark(
    lambda: emb_onnx.run(None, {"fbank": dummy_mel}),
    "ONNX  embedding"
)
results["emb_opv"] = benchmark(
    lambda: emb_ov({"fbank": dummy_mel}),
    "OpenVINO  embedding"
)


── Embedding (1s chunk · input: 1×98×80) ──
  ONNX  embedding                     mean=19.39ms  std=1.75ms  min=16.39ms  max=48.72ms
  OpenVINO  embedding                 mean=23.90ms  std=2.17ms  min=21.08ms  max=74.73ms


In [63]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("\n── Summary ──")
print(f"  {'Model':<35} {'ONNX mean':>12} {'OV mean':>12} {'Speedup':>10}")
print(f"  {'─'*70}")

for name in ["seg", "emb"]:
    onnx_mean = results[f"{name}_onnx"].mean()
    ov_mean   = results[f"{name}_opv"].mean()
    speedup   = onnx_mean / ov_mean
    label     = "Segmentation" if name == "seg" else "Embedding"
    winner    = "OV faster ✓" if speedup > 1 else "ONNX faster ✓"
    print(f"  {label:<35} {onnx_mean:>10.2f}ms {ov_mean:>10.2f}ms   {speedup:>5.2f}x  {winner}")


── Summary ──
  Model                                  ONNX mean      OV mean    Speedup
  ──────────────────────────────────────────────────────────────────────
  Segmentation                             40.59ms      48.40ms    0.84x  ONNX faster ✓
  Embedding                                19.39ms      23.90ms    0.81x  ONNX faster ✓
